# Letter Recognition: From Exploration to a Reproducible Model

Author: Lucy Giang (Mar. 2026)

## Overview

This project predicts one of 26 uppercase letters from 16 numeric measurements describing a character image. The data contains 20,000 labeled examples. The goal is to build a reproducible classification workflow, compare different model families, and inspect the errors that remain.

We reserve a stratified test set for one final, unbiased comparison. Model selection and tuning use only the training split with stratified cross-validation. Run the notebook from the project root so the relative data path resolves.


### Setup

The helpers below centralize repeated evaluation steps. Search spaces are intentionally compact: they cover meaningful choices while keeping a full notebook run practical on a laptop.


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib

RANDOM_STATE = 42
LABELS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)


## Exploratory data analysis

We first confirm the table structure and quality, then inspect representative feature distributions, relationships among all measurements, and the balance across all letter classes. This helps identify scale differences and potential modeling considerations before fitting anything.


In [ ]:
DATA_PATH = Path("data/letterrecognition/letter-recognition.data")
COLUMNS = [
    "letter", "x_box", "y_box", "width", "height", "on_pixels",
    "x_bar", "y_bar", "x2_bar", "y2_bar", "xy_bar", "x2_y_br",
    "xy2_br", "x_edge", "x_edge_y", "y_edge", "y_edge_x"
]
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")
data = pd.read_csv(DATA_PATH, header=None, names=COLUMNS)
features = COLUMNS[1:]
X = data[features]
y = data["letter"]

print(f"Rows: {len(data):,} | Features: {len(features)} | Classes: {y.nunique()}")
display(data.head())
display(data[features].describe().T)
print("Missing values:", int(data.isna().sum().sum()))
print("Duplicate rows:", int(data.duplicated().sum()))


The dataset has the expected 16 numeric inputs and 26 target labels. The summary table gives a first view of feature ranges; the plots below make skew and scale differences easier to see. No preprocessing is learned from the full dataset: any scaling required by a model will be fitted inside its pipeline during cross-validation.


In [ ]:
distribution_features = ["width", "height", "on_pixels", "x_bar", "y_bar", "x_edge"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for feature, ax in zip(distribution_features, axes.flat):
    sns.histplot(data=data, x=feature, kde=True, ax=ax, color="teal")
    ax.set_title(f"Distribution of {feature}")
plt.tight_layout()
plt.show()


The selected measurements have different ranges and shapes, so scale-sensitive methods such as SVC and MLP should standardize inputs. Tree models do not require that transformation. Distribution tails also remind us that an average score alone may hide difficult subsets of the data.


In [ ]:
corr = X.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="vlag", center=0, square=True, linewidths=0.2, cbar_kws={"shrink": 0.75})
plt.title("Feature correlation matrix (16 numeric features)")
plt.tight_layout()
plt.show()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = (upper.stack().abs().sort_values(ascending=False).head(8).rename("absolute correlation").to_frame())
display(strong_pairs)


Correlation highlights pairs with overlapping linear information; it does not by itself establish that a feature should be removed. We keep all 16 measurements for now and compare models that handle correlated inputs differently.


In [ ]:
letter_counts = y.value_counts().reindex(LABELS)
fig, ax = plt.subplots(figsize=(13, 4))
letter_counts.plot(kind="bar", ax=ax, color="slateblue")
ax.set(title="Class distribution across all 26 letters", xlabel="Letter", ylabel="Examples")
ax.set_xticklabels(LABELS, rotation=0)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"examples": letter_counts, "share": (letter_counts / len(y)).map(lambda v: f"{v:.2%}")}))
print(f"Smallest / largest class: {letter_counts.min()} / {letter_counts.max()}")


All classes are represented and the bar chart lets us check whether any class is unusually small. We preserve each class's proportion in both the holdout split and every CV fold, making comparisons less sensitive to accidental class imbalance.


## Data preprocessing and pipeline construction

We create the test set once and keep it untouched during all model choices. Stratified three-fold cross-validation on the remaining data will supply the validation estimate. Scaling is placed inside each relevant pipeline, preventing information from a validation fold from leaking into its scaler.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print(f"Training: {len(X_train):,} | Test: {len(X_test):,}")
print("Train proportions match full data:", np.allclose(
    y_train.value_counts(normalize=True).sort_index().values,
    y.value_counts(normalize=True).sort_index().values, atol=0.01
))


## Baseline, model training, and hyperparameter tuning

We compare a majority-class DummyClassifier, a Random Forest, an RBF Support Vector Classifier, and a multilayer perceptron. Each search is scored by accuracy across the same stratified folds. The standardized pipelines are essential for SVC and MLP; the forest uses raw feature values. The reported validation accuracy is the best mean CV score selected without accessing the test set.


In [ ]:
def tune_and_record(name, estimator, param_grid):
    """Fit a small grid on training data and return its CV-selected estimator and metrics."""
    search = GridSearchCV(
        estimator=estimator, param_grid=param_grid, scoring="accuracy",
        cv=cv, n_jobs=-1, refit=True, return_train_score=False, verbose=0
    )
    started = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - started
    return {
        "name": name, "estimator": search.best_estimator_,
        "validation_accuracy": search.best_score_, "test_accuracy": accuracy_score(y_test, search.predict(X_test)),
        "training_time_seconds": elapsed, "best_params": search.best_params_,
        "search": search
    }

results = []
baseline = DummyClassifier(strategy="most_frequent")
started = time.perf_counter()
baseline_cv = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
baseline.fit(X_train, y_train)
baseline_elapsed = time.perf_counter() - started
results.append({
    "name": "Dummy (most frequent)", "estimator": baseline,
    "validation_accuracy": baseline_cv.mean(),
    "test_accuracy": accuracy_score(y_test, baseline.predict(X_test)),
    "training_time_seconds": baseline_elapsed, "best_params": {}, "search": None
})

rf = tune_and_record(
    "Random Forest", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    {"n_estimators": [250], "max_features": ["sqrt", 0.7], "min_samples_leaf": [1, 2]}
)
results.append(rf)
svc = tune_and_record(
    "RBF SVC", make_pipeline(StandardScaler(), SVC()),
    {"svc__C": [3, 10], "svc__gamma": ["scale", 0.03]}
)
results.append(svc)
mlp = tune_and_record(
    "MLP", make_pipeline(StandardScaler(), MLPClassifier(
        max_iter=120, early_stopping=True, n_iter_no_change=8,
        random_state=RANDOM_STATE, batch_size=256
    )),
    {"mlpclassifier__hidden_layer_sizes": [(128, 64), (100,)],
     "mlpclassifier__alpha": [0.0001, 0.001],
     "mlpclassifier__learning_rate_init": [0.001]}
)
results.append(mlp)

for item in results:
    print(f"{item['name']}: CV={item['validation_accuracy']:.3%}, test={item['test_accuracy']:.3%}, time={item['training_time_seconds']:.1f}s")


The searches select settings using training folds only, then refit the winning configuration on the full training split. Training time includes cross-validation and refitting for searched models; the baseline time covers its CV and final fit. This makes time comparisons operationally meaningful for this notebook run, though timings vary by hardware.


### Model comparison

The table puts cross-validated validation accuracy beside the final holdout accuracy. A small gap is expected because the holdout is a separate sample; a large gap would warrant checking variance or overfitting. Training time is the time spent selecting and fitting each candidate.


In [ ]:
comparison = pd.DataFrame([{
    "Model": r["name"],
    "Training time (s)": r["training_time_seconds"],
    "Validation accuracy (CV)": r["validation_accuracy"],
    "Test accuracy": r["test_accuracy"]
} for r in results]).sort_values("Validation accuracy (CV)", ascending=False).reset_index(drop=True)
display(comparison.style.format({
    "Training time (s)": "{:.1f}",
    "Validation accuracy (CV)": "{:.2%}",
    "Test accuracy": "{:.2%}"
}))
print("Selected hyperparameters:")
for r in results:
    if r["best_params"]:
        print(f"{r['name']}: {r['best_params']}")


## Model evaluation and error analysis

We select the model with the highest CV validation accuracy, then inspect its held-out predictions. The test set is used here for final reporting and diagnosis, not for another round of tuning. The selected model and its test predictions are kept together for the remaining cells.


In [ ]:
best_result = max(results, key=lambda item: item["validation_accuracy"])
best_model = best_result["estimator"]
test_predictions = best_model.predict(X_test)
print(f"Selected model: {best_result['name']}")
print(f"CV accuracy: {best_result['validation_accuracy']:.2%} | Test accuracy: {accuracy_score(y_test, test_predictions):.2%}")
cm = confusion_matrix(y_test, test_predictions, labels=LABELS)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues", xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False)
ax.set(title=f"Test confusion matrix — {best_result['name']}", xlabel="Predicted letter", ylabel="True letter")
plt.tight_layout()
plt.show()


The confusion matrix separates systematic confusions from general error volume. The table below ranks directed true-to-predicted mistakes, so (for example) A→R and R→A are counted separately.


In [ ]:
errors = pd.DataFrame({"true": y_test.to_numpy(), "predicted": test_predictions})
errors = errors[errors["true"] != errors["predicted"]]
top_confusions = (errors.groupby(["true", "predicted"]).size()
                  .sort_values(ascending=False).head(12)
                  .rename("count").reset_index())
display(top_confusions)


Per-class precision measures how often a predicted letter is correct; recall measures how many true examples of that letter are found. The lowest values can point to visually similar shapes or measurements that do not capture the distinction well. These are diagnostic clues rather than proof of a specific cause.


In [ ]:
report = classification_report(y_test, test_predictions, labels=LABELS, output_dict=True, zero_division=0)
per_class = pd.DataFrame(report).T.loc[LABELS, ["precision", "recall", "f1-score", "support"]]
display(per_class.sort_values("recall").head(8).style.format({"precision":"{:.1%}", "recall":"{:.1%}", "f1-score":"{:.1%}", "support":"{:.0f}"}))
print("Lowest precision classes:")
display(per_class.sort_values("precision").head(5).style.format({"precision":"{:.1%}", "recall":"{:.1%}", "f1-score":"{:.1%}", "support":"{:.0f}"}))


Read these rows alongside the top confusions: a low-recall class is often being assigned to one or two neighboring classes, while low precision means other letters are being mistaken for it. Geometric and pixel-density summaries compress the original image, so strokes or small shape differences may no longer be distinguishable from the 16 measurements alone.


## Conclusion

The workflow now checks feature distributions and correlations, verifies the class balance, compares four model families, tunes three of them with stratified CV, and reports per-class errors on a held-out test set. The CV-selected model is the candidate artifact. Before deployment, the consuming application should validate the 16 input feature names and order and monitor performance on newly labeled examples.

The serialized object includes any preprocessing pipeline, so inference uses the same fitted scaling as training. The data path below is relative to the project root; the artifact is written to `artifacts/`. This repository will include my saved artifact from my local run but not the folder.


In [ ]:
ARTIFACT_PATH = Path("artifacts/letter_recognition_model.joblib")
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": best_model,
    "feature_columns": features,
    "classes": LABELS,
    "selected_model": best_result["name"],
    "random_state": RANDOM_STATE
}, ARTIFACT_PATH)
print(f"Saved deployable model bundle to: {ARTIFACT_PATH.resolve()}")


### Example inference

Load the bundle and pass a DataFrame with the same 16 named inputs. Keeping feature names in the artifact makes the expected input contract explicit and helps prevent accidental column reordering.


In [ ]:
loaded_bundle = joblib.load(ARTIFACT_PATH)
loaded_model = loaded_bundle["model"]
example_batch = X_test.iloc[:3][loaded_bundle["feature_columns"]]
pd.DataFrame({"predicted_letter": loaded_model.predict(example_batch)}, index=example_batch.index)
